# 03_03_SGD_Momentum_Adam_の比較

コード：`ch03/blocks/03_03_SGD_Momentum_Adam_の比較.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 細長い谷 L = 0.05x² + y² で、SGD・Momentum・Adam を比べる
import numpy as np

def loss(p):
    return 0.05 * p[0] ** 2 + p[1] ** 2

def grad(p):
    return np.array([0.1 * p[0], 2.0 * p[1]])

def sgd(p, g, s, lr):
    return p - lr * g

def momentum(p, g, s, lr, beta=0.8):
    s["v"] = beta * s.get("v", 0) - lr * g          # 速度に勾配を積む
    return p + s["v"]

def adam(p, g, s, lr, b1=0.9, b2=0.999, eps=1e-8):
    s["t"] = s.get("t", 0) + 1
    s["m"] = b1 * s.get("m", 0) + (1 - b1) * g       # 1 次モーメント（慣性）
    s["u"] = b2 * s.get("u", 0) + (1 - b2) * g * g   # 2 次モーメント（大きさ）
    m_hat = s["m"] / (1 - b1 ** s["t"])
    u_hat = s["u"] / (1 - b2 ** s["t"])
    return p - lr * m_hat / (np.sqrt(u_hat) + eps)

BASE = {sgd: 0.8, momentum: 0.8, adam: 0.3}          # スケール 1.0 のときの学習率

def run(opt, scale, steps=100):
    p, s, hist = np.array([-9.0, 2.0]), {}, []
    for t in range(steps + 1):
        hist.append(loss(p))
        p = opt(p, grad(p), s, BASE[opt] * scale)
    return hist

for scale in (1.0, 1.3):
    print(f"学習率スケール {scale}")
    for opt in (sgd, momentum, adam):
        h = run(opt, scale)
        reach = next((t for t, L in enumerate(h) if L < 1e-3), None)
        cells = "  ".join(f"{h[t]:9.3g}" for t in (0, 5, 10, 20, 50, 100))
        print(f"  {opt.__name__:8s} {cells}   L<1e-3 到達: {reach}")

# 軌跡の最初の 4 歩（スケール 1.0）
for opt in (sgd, momentum, adam):
    p, s, pts = np.array([-9.0, 2.0]), {}, []
    for t in range(4):
        p = opt(p, grad(p), s, BASE[opt])
        pts.append(f"({p[0]:5.2f}, {p[1]:5.2f})")
    print(f"  {opt.__name__:8s} " + " → ".join(pts))
